# OCR RX 580 + CPU — versão revisada V3 (com interface web local)

OCR local para PDFs e imagens em **CPU**, rota padrão **rapidocr (ONNX Runtime puro)**;
`paddleocr` 3.x é alternativa opcional. A RX 580 **não acelera este OCR** — GPU fica só
para VLM via Vulkan como etapa separada, nunca substituindo automaticamente o texto.

## Privacidade e limitações

- **Privacidade**: todo o processamento é local. Documentos e resultados não são
  enviados à internet; a interface web faz bind apenas em `127.0.0.1` com `share=False`.
- **Limitações**: OCR de tabelas é experimental (o CSV reflete linhas reconhecidas,
  não uma estrutura tabular detectada); reconhecimento de **fórmulas não está
  implementado** até existir módulo opcional testado; processamento é **sequencial**
  para preservar os 16 GB de RAM; deskew cobre inclinações até ±12°.

## Instalação

```bash
python3 -m venv .venv && source .venv/bin/activate
pip install -r ocr_v3/requirements.txt     # rapidocr, onnxruntime, gradio etc.
jupyter notebook
```

> A célula seguinte também verifica/installa o que falta, de forma idempotente.
> Se o backend paddle falhar ao iniciar, a V3 cai automaticamente para rapidocr.


In [ ]:
# Verificação IDEMPOTENTE de dependências — instala SOMENTE o que estiver faltando.
import importlib.util
import subprocess
import sys

REQUERIDOS = {
    "numpy": "numpy>=1.26",
    "cv2": "opencv-python-headless>=4.9",
    "pymupdf": "pymupdf>=1.24",
    "rapidocr": "rapidocr>=3.9",
    "onnxruntime": "onnxruntime>=1.17",
    "tqdm": "tqdm>=4.66",
    "gradio": "gradio>=6.25",
}

_faltando = [pkg for modulo, pkg in REQUERIDOS.items() if importlib.util.find_spec(modulo) is None]
if _faltando:
    print("Instalando:", ", ".join(_faltando))
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *_faltando], check=True)
    print("Instalação concluída.")
else:
    print("Dependências OK — nada a instalar.")


In [ ]:
# Diagnóstico de hardware (apenas informativo).
# O OCR roda em CPU / ONNX Runtime; a RX 580 não acelera este pipeline.
import platform
import sys
from pathlib import Path


def _cpu():
    try:
        for linha in Path("/proc/cpuinfo").read_text(encoding="utf-8").splitlines():
            if linha.lower().startswith("model name"):
                return linha.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor() or "desconhecida"


def _ram_disponivel():
    try:
        for linha in Path("/proc/meminfo").read_text(encoding="utf-8").splitlines():
            if linha.startswith("MemAvailable"):
                return f"{int(linha.split()[1]) / 1024**2:.1f} GiB livres"
    except Exception:
        pass
    return "indisponível"


def _gpu_informativa():
    """Detecção best-effort e HONESTA: nenhuma GPU é usada pelo OCR."""
    mapa = {"0x1002": "AMD", "0x10de": "NVIDIA", "0x8086": "Intel"}
    try:
        achadas = set()
        for card in sorted(Path("/sys/class/drm").glob("card[0-9]*")):
            vend = card / "device" / "vendor"
            if vend.exists():
                nome = mapa.get(vend.read_text().strip().lower())
                if nome:
                    achadas.add(nome)
        return ", ".join(sorted(achadas)) or "nenhuma detectada"
    except Exception:
        return "desconhecida"


print(f"Python       : {sys.version.split()[0]} ({platform.machine()})")
try:
    from importlib.metadata import version
    print(f"RapidOCR     : {version('rapidocr')}")
    print(f"ONNX Runtime : {version('onnxruntime')}")
except Exception:
    pass
try:  # versão do projeto sem depender da ordem das células
    from ocr_v3.ocr_v3 import _VERSAO as _ver_projeto
except Exception:
    _ver_projeto = "?"
print(f"CPU          : {_cpu()}")
print(f"RAM          : {_ram_disponivel()}")
print(f"GPU (info)   : {_gpu_informativa()} — não usada pelo OCR")
print(f"Backend OCR  : CPU / ONNX Runtime | projeto: OCR Local V{_ver_projeto}")


In [ ]:
# OCR local V3 — revisão do notebook "OCR RX 580 + CPU V2"
#
# Rota padrão: rapidocr (ONNX Runtime puro em CPU) ou PaddleOCR 3.x.
# A RX 580 não acelera este OCR; GPU fica apenas para VLM via Vulkan,
# como etapa separada — nunca substituindo automaticamente o texto extraído.
#
# Principais correções vs V2 (detalhes no README.md):
#   [BUG-1] Cache quente não destrói mais o PDF pesquisável (imagens reidratadas).
#   [BUG-2] Memória: páginas viram buffer JPEG (~1-2 MB) em vez de ndarray cru (~26 MB);
#           buffers são liberados após exportação; resultados não acumulam imagem.
#   [MED-1] Schema PaddleOCR 3.x: desembrulha {"res": {...}} e aceita atributos.
#   [MED-2] Construtor PaddleOCR tolerante: tenta engine="onnxruntime" e degrada sem quebrar.
#   [MED-3] Deskew corrigido para OpenCV >= 4.5 (convenção de ângulo do minAreaRect mudou).
#   [MIN-*] Camada de texto via insert_text (nunca recorta); páginas com erro ganham
#           placeholder (numeração preservada); temporários sempre limpos; assinatura de
#           cache ignora flags só de exportação; ID por CONTEÚDO do arquivo; saída respeita
#           cfg.saida (V2 usava variável global); fallback automático rapidocr; barra de
#           progresso opcional; grayscale de PDF tratado.

from __future__ import annotations

from collections.abc import Callable
from contextlib import contextmanager
from dataclasses import dataclass, asdict, replace
from pathlib import Path
import csv
import hashlib
import io
import json
import logging
import math
import os
import sys
import tempfile
import time
from datetime import datetime

import cv2
import numpy as np

try:  # pymupdf novo nome oficial; cai para o alias antigo em versões velhas
    import pymupdf as fitz
except ImportError:
    import fitz


_VERSAO = "3.1.0"

try:
    from tqdm.auto import tqdm as _tqdm_auto
except ImportError:  # tqdm é opcional
    def _tqdm_auto(iterable, **_):
        return iterable


def tqdm(iterable, **kw):
    """Barra de progresso que se desliga sozinha fora de TTY/notebook (logs limpos em CI)."""
    if "disable" not in kw:
        em_notebook = False
        try:
            em_notebook = get_ipython is not None and "get_ipython" in dir(sys.modules.get("__main__", {}))  # noqa: F821
        except Exception:
            pass
        kw["disable"] = not (sys.stdout.isatty() or em_notebook)
    return _tqdm_auto(iterable, **kw)


# Nível de log configurável sem tocar no código:  export OCR_V3_LOG=WARNING
logging.basicConfig(
    level=getattr(logging, os.environ.get("OCR_V3_LOG", "INFO").upper(), logging.INFO),
    format="%(asctime)s | %(levelname)s | %(message)s",
)
LOG = logging.getLogger("ocr_v3")

_VERSAO_SCHEMA_CACHE = "ocr-v3"

# Callback opcional de progresso: (atual, total, mensagem). Usado pela UI;
# CLI e testes não passam nada e continuam com tqdm/como antes.
ProgressoCallback = Callable[[int, int, str], None]


# --------------------------------------------------------------------------- #
# Configuração
# --------------------------------------------------------------------------- #

@dataclass(frozen=True)
class ConfigV3:
    entrada: str = "documentos"
    saida: str = "saida_ocr_v3"
    backend: str = "rapidocr"  # rapidocr (padrão: ONNX Runtime puro) | paddle
    idioma: str = "pt"
    dpi: int = 300
    max_pixels_pagina: int = 18_000_000
    confianca_minima: float = 0.45
    pre_processar: bool = True
    deskew: bool = True
    denoise: bool = False
    cache: bool = True
    gerar_pdf_pesquisavel: bool = True
    jpeg_qualidade: int = 88  # afeta só o PDF pesquisável, não o OCR
    extensoes: tuple = (".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp")


CFG3 = ConfigV3()


# --------------------------------------------------------------------------- #
# Escrita atômica (temporários sempre limpos, mesmo em exceção)
# --------------------------------------------------------------------------- #

@contextmanager
def _escrita_atomica(path: Path):
    """Yield um arquivo temporário no mesmo diretório do destino e o promove via replace()."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    f = tempfile.NamedTemporaryFile("wb", dir=path.parent, delete=False, suffix=".tmp")
    tmp = Path(f.name)
    try:
        yield f
        f.flush()
        os.fsync(f.fileno())  # garante dados no disco antes do replace (quedas de energia)
        f.close()
        tmp.replace(path)
    finally:
        if tmp.exists():
            tmp.unlink(missing_ok=True)


def _atomic_json(path: Path, obj) -> None:
    with _escrita_atomica(path) as f:
        f.write(json.dumps(obj, ensure_ascii=False, indent=2).encode("utf-8"))


def _atomic_bytes(path: Path, dados: bytes) -> None:
    with _escrita_atomica(path) as f:
        f.write(dados)


# --------------------------------------------------------------------------- #
# Identidade e assinatura de cache
# --------------------------------------------------------------------------- #

def _hash_arquivo(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while bloco := fh.read(chunk):
            h.update(bloco)
    return h.hexdigest()


def _id(path: Path) -> str:
    """ID por conteúdo: mover/renomear a pasta NÃO invalida o cache."""
    path = Path(path)
    return f"{path.stem}_{_hash_arquivo(path)[:12]}"


# Campos que não alteram a inferência — mudanças neles não devem invalidar o cache.
_CAMPOS_SOMENTE_EXPORT = frozenset(
    {"entrada", "saida", "gerar_pdf_pesquisavel", "jpeg_qualidade", "cache"}
)


def _signature(path: Path, cfg: ConfigV3) -> str:
    st = Path(path).stat()
    d = asdict(cfg)
    for k in _CAMPOS_SOMENTE_EXPORT:
        d.pop(k, None)
    payload = [str(path.resolve()), st.st_size, st.st_mtime_ns, _VERSAO_SCHEMA_CACHE, d]
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=list).encode("utf-8")
    ).hexdigest()


_CHAVES_PESADAS = frozenset({"jpg", "imagem"})


def _leve(resultado: dict) -> dict:
    """Cópia sem buffers de imagem nem chaves internas (para JSON/cache)."""
    paginas = [
        {k: v for k, v in p.items() if k not in _CHAVES_PESADAS and not k.startswith("_")}
        for p in resultado["paginas"]
    ]
    return {k: v for k, v in {**resultado, "paginas": paginas}.items() if not k.startswith("_")}


# --------------------------------------------------------------------------- #
# Entrada: imagens soltas e páginas de PDF
# --------------------------------------------------------------------------- #

def _load_image(path: Path) -> np.ndarray:
    img = cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Imagem inválida: {path.name}")
    if max(img.shape[:2]) > 16000:
        raise ValueError("Imagem recusada: lado acima de 16000 pixels")
    return img


def _pix_to_bgr(pix: "fitz.Pixmap") -> np.ndarray:
    raw = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)
    if pix.n == 4:
        return cv2.cvtColor(raw, cv2.COLOR_RGBA2BGR)
    if pix.n == 3:
        return cv2.cvtColor(raw, cv2.COLOR_RGB2BGR)
    if pix.n == 1:
        return cv2.cvtColor(raw, cv2.COLOR_GRAY2BGR)
    raise ValueError(f"Pixmap com {pix.n} canais não suportado")


def _pdf_pages(path: Path, cfg: ConfigV3):
    """Gerador preguiçoso: (numero, imagem_bgr, dpi_real). Não mantém o documento aberto entre páginas."""
    doc = fitz.open(str(path))
    try:
        if doc.needs_pass:
            raise ValueError("PDF protegido por senha")
        for n, page in enumerate(doc, 1):
            scale = cfg.dpi / 72
            pixels = page.rect.width * page.rect.height * scale * scale
            if pixels > cfg.max_pixels_pagina:
                scale *= math.sqrt(cfg.max_pixels_pagina / pixels)
            pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
            try:
                yield n, _pix_to_bgr(pix), round(scale * 72, 1)
            finally:
                pix = None  # libera o pixmap assim que a página sai do laço
    finally:
        doc.close()


def _fonte_paginas(path: Path, cfg: ConfigV3):
    """Uniformiza PDFs e imagens: iterável de (numero, imagem_bgr, dpi_real)."""
    if Path(path).suffix.lower() == ".pdf":
        return _pdf_pages(path, cfg)
    return [(1, _load_image(Path(path)), None)]


def _conta_paginas(path: Path) -> int | None:
    if Path(path).suffix.lower() != ".pdf":
        return 1
    try:
        with fitz.open(str(path)) as doc:
            return doc.page_count
    except Exception:
        return None


# --------------------------------------------------------------------------- #
# Pré-processamento (deskew compatível com OpenCV >= 4.5)
# --------------------------------------------------------------------------- #

def _medir_skew(img_bgr: np.ndarray) -> float:
    """Inclinação dominante em graus, faixa (-45, 45].

    OpenCV >= 4.5 retorna o ângulo do minAreaRect em (0, 90] — a fórmula antiga
    ``-(90+a) if a < -45`` nunca disparava e o deskew ficava intermitente.
    Aqui normalizamos referenciando sempre o lado maior (a linha de texto).
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    binaria = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    pts = np.column_stack(np.where(binaria > 0))
    if len(pts) < 300:
        return 0.0
    (_cx, _cy), (rw, rh), bruto = cv2.minAreaRect(pts)
    if rw < rh:
        bruto += 90  # passa a medir em relação ao lado maior
    return ((bruto + 45) % 90) - 45


def _pre(img: np.ndarray, cfg: ConfigV3) -> np.ndarray:
    if not cfg.pre_processar:
        return img
    out = img
    if cfg.deskew:
        skew = _medir_skew(out)
        if 0.35 <= abs(skew) <= 12:
            h, w = out.shape[:2]
            out = cv2.warpAffine(
                out,
                cv2.getRotationMatrix2D((w / 2, h / 2), -skew, 1),
                (w, h),
                flags=cv2.INTER_CUBIC,
                borderMode=cv2.BORDER_REPLICATE,
            )
    if max(out.shape[:2]) < 1400:
        s = 1400 / max(out.shape[:2])
        out = cv2.resize(out, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
    if cfg.denoise:
        out = cv2.fastNlMeansDenoisingColored(out, None, 4, 4, 7, 21)
    return out


# --------------------------------------------------------------------------- #
# Ordem de leitura (implementação pura, sem mutar os dicts de entrada)
# --------------------------------------------------------------------------- #

def _order(items: list[dict]) -> list[dict]:
    if not items:
        return []
    decorados = []
    for it in items:
        b = np.asarray(it["box"], float)
        cy = float(b[:, 1].mean())
        cx = float(b[:, 0].mean())
        altura = max(1.0, float(np.ptp(b[:, 1])))
        decorados.append((cy, cx, altura, it))
    banda = max(8.0, float(np.median([d[2] for d in decorados])))
    decorados.sort(key=lambda d: (round(d[0] / banda), d[1]))
    return [d[3] for d in decorados]


# --------------------------------------------------------------------------- #
# Motores OCR
# --------------------------------------------------------------------------- #

_CAMPOS_PADDLE = ("rec_texts", "rec_scores", "rec_polys")


class MotorOCRV3:
    def __init__(self, cfg: ConfigV3):
        self.cfg = cfg
        if cfg.backend == "paddle":
            self._init_paddle(cfg)
        elif cfg.backend == "rapidocr":
            self._init_rapidocr(cfg)
        else:
            raise ValueError("backend deve ser 'rapidocr' ou 'paddle'")

    # -- paddle ------------------------------------------------------------- #

    def _init_paddle(self, cfg: ConfigV3) -> None:
        from paddleocr import PaddleOCR

        base = dict(
            lang=cfg.idioma,
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=True,
        )
        try:  # rota ONNX Runtime quando a versão instalada suporta o parâmetro
            self.engine = PaddleOCR(device="cpu", engine="onnxruntime", **base)
            self.sabor = "paddle-onnx"
        except TypeError:  # versões sem esses kwargs: usa o default da instalação
            LOG.warning("PaddleOCR sem kwargs 'device/engine'; usando defaults da versão instalada")
            self.engine = PaddleOCR(**base)
            self.sabor = "paddle"

    @staticmethod
    def _desembrulhar_paddle(obj):
        """Aceita .json (propriedade ou callable), dict plano, dict aninhado {'res': {...}}
        (1+ níveis) ou objeto com atributos rec_texts/rec_scores/rec_polys."""
        d = getattr(obj, "json", None)
        if callable(d):
            d = d()
        if d is None:
            d = obj
        for _ in range(3):
            if isinstance(d, dict) and all(k in d for k in _CAMPOS_PADDLE):
                return (d["rec_texts"], d["rec_scores"], d["rec_polys"])
            proximo = d.get("res") if isinstance(d, dict) else None
            if isinstance(proximo, dict):
                d = proximo
                continue
            break
        textos = getattr(obj, "rec_texts", None)
        if textos is not None:
            return (
                textos,
                getattr(obj, "rec_scores"),
                getattr(obj, "rec_polys"),
            )
        chaves = list(d) if isinstance(d, dict) else type(d).__name__
        raise RuntimeError(f"Schema PaddleOCR inesperado: {chaves}")

    # -- rapidocr ------------------------------------------------------------ #

    def _init_rapidocr(self, cfg: ConfigV3) -> None:
        from rapidocr import RapidOCR

        self.engine = RapidOCR()
        self.sabor = "rapidocr-onnx"

    @staticmethod
    def _adaptar_rapidocr(saida) -> list[tuple]:
        boxes = getattr(saida, "boxes", None)  # rapidocr >= 2.x
        if boxes is not None:
            txts = getattr(saida, "txts", None) or []
            scores = getattr(saida, "scores", None) or []
            return list(zip(boxes, txts, scores))
        res = getattr(saida, "results", saida)  # rapidocr 1.x: tupla (lista, tempos)
        if isinstance(res, tuple):
            res = res[0]
        return [(x[0], x[1], x[2]) for x in (res or [])]

    # -- interface comum ------------------------------------------------------ #

    def inferir(self, img: np.ndarray) -> list[dict]:
        if self.cfg.backend == "rapidocr":
            bruto = self._adaptar_rapidocr(self.engine(img))
        else:
            res = list(self.engine.predict(img))
            if not res:
                return []
            bruto = zip(*self._desembrulhar_paddle(res[0]))
        itens = [
            {
                "box": np.asarray(b, float).reshape(-1, 2).tolist(),
                "texto": str(t),
                "score": float(s),
            }
            for b, t, s in bruto
        ]
        return _order(itens)


# --------------------------------------------------------------------------- #
# Processamento por documento (memória controlada)
# --------------------------------------------------------------------------- #

def processar_v3(
    path, cfg: ConfigV3, motor: MotorOCRV3, progresso: ProgressoCallback | None = None
) -> dict:
    path = Path(path)
    out_dir = Path(cfg.saida)
    base = out_dir / _id(path)
    cache_path = base.with_suffix(".cache.json")
    sig = _signature(path, cfg)

    if cfg.cache and cache_path.exists():
        try:
            antigo = json.loads(cache_path.read_text(encoding="utf-8"))
            if antigo.get("assinatura") == sig:
                r = antigo["resultado"]
                r["_do_cache"] = True
                LOG.info("Cache válido para %s (inferência pulada)", path.name)
                return r
        except Exception:
            LOG.warning("Cache ilegível para %s; reprocessando", path.name)

    total = _conta_paginas(path)
    paginas: list[dict] = []
    erros: list[dict] = []
    iniciado = time.perf_counter()

    for n, img, dpi_real in tqdm(_fonte_paginas(path, cfg), total=total, desc=path.name[:40]):
        if progresso is not None:  # mensagem real de página (UI); nunca derruba o pipeline
            msg = (
                f"Processando {path.name} — página {n} de {total}"
                if total
                else f"Processando {path.name} — página {n}"
            )
            try:
                progresso(n, total or 0, msg)
            except Exception:
                LOG.debug("Callback de progresso falhou (ignorado)", exc_info=True)
        pagina: dict = {
            "pagina": n,
            "dpi_real": dpi_real,
            "largura_px": None,
            "altura_px": None,
            "itens": [],
            "texto_ocr": "",
            "confianca_media": 0.0,
        }
        try:
            usado = _pre(img, cfg)
            del img  # libera a página crua antes da inferência
            itens = [
                z
                for z in motor.inferir(usado)
                if z["score"] >= cfg.confianca_minima
            ]
            scores = [z["score"] for z in itens]
            ok, buf = cv2.imencode(
                ".jpg", usado, [int(cv2.IMWRITE_JPEG_QUALITY), int(cfg.jpeg_qualidade)]
            )
            if not ok:
                raise RuntimeError("Falha ao codificar JPEG da página")
            pagina.update(
                {
                    "largura_px": int(usado.shape[1]),
                    "altura_px": int(usado.shape[0]),
                    "itens": itens,
                    "texto_ocr": "\n".join(z["texto"] for z in itens),
                    "confianca_media": round(float(np.mean(scores)), 4) if scores else 0.0,
                    "jpg": buf.tobytes(),  # ~1-2 MB; o ndarray cru (~26 MB) já foi liberado
                }
            )
            del usado, buf
        except Exception as e:  # isolamento por página + placeholder preserva numeração
            LOG.exception("Falha em %s, página %d", path.name, n)
            pagina["erro"] = f"{type(e).__name__}: {e}"
            erros.append({"pagina": n, "erro": pagina["erro"]})
        paginas.append(pagina)

    resultado = {
        "arquivo_origem": str(path),
        "id": _id(path),
        "tempo_s": round(time.perf_counter() - iniciado, 3),
        "paginas": paginas,
        "erros": erros,
    }
    if cfg.cache:
        _atomic_json(cache_path, {"assinatura": sig, "resultado": _leve(resultado)})
    return resultado


# --------------------------------------------------------------------------- #
# Exportação
# --------------------------------------------------------------------------- #

_TAMANHO_PLACEHOLDER = (595, 842)  # A4 em pontos, para páginas que falharam


# A camada invisível usa a fonte base Helvetica (encoding Latin-1/WinAnsi), que NÃO
# cobre vários glifos Unicode — o MuPDF os troca silenciosamente por "·" (ponto médio),
# corrompendo a busca. Normalizamos a pontuação tipográfica para ASCII e descartamos
# o que é genuinamente não-encodável (emoji etc.), preservando a integridade das palavras.
_TRADUCOES_CAMADA = str.maketrans({
    "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u201f": '"',   # aspas curvas duplas
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'",   # aspas curvas simples
    "\u00ab": '"', "\u00bb": '"',                                  # «»
    "\u2013": "-", "\u2014": "-", "\u2015": "-", "\u2012": "-",    # traços
    "\u2026": "...", "\u00a0": " ", "\u200b": "",                  # reticências/nbsp/zero-width
    "\u00b7": " ",                                                 # · (placeholder de glifo não reconhecido do OCR)
})


def _sanitizar_camada(texto: str) -> str:
    t = str(texto).translate(_TRADUCOES_CAMADA)
    try:
        t.encode("cp1252")
        return t
    except UnicodeEncodeError:
        return "".join(c for c in t if c.encode("cp1252", "ignore"))


def _quebrar(texto: str, max_chars: int) -> list[str]:
    """Quebra por palavras; tokens gigantescos são cortados à força."""
    linhas: list[str] = []
    atual = ""
    for palavra in texto.split():
        candidato = f"{atual} {palavra}".strip()
        if len(candidato) <= max_chars:
            atual = candidato
            continue
        if atual:
            linhas.append(atual)
        while len(palavra) > max_chars:  # token sem espaço maior que a página
            linhas.append(palavra[:max_chars])
            palavra = palavra[max_chars:]
        atual = palavra
    if atual:
        linhas.append(atual)
    return linhas


def _inserir_camada_texto(page: "fitz.Page", itens: list[dict]) -> None:
    """Camada invisível completa: nada é descartado.

    MuPDF não extrai (nem busca) glifos fora da página — insert_textbox recortava
    caixas baixas e uma linha longa transbordando a largura também sumiria.
    Aqui o texto é re-fluxado em linhas dentro da página, empilhando para baixo e
    recomeçando do topo se estourar a altura (posição aproximada, conteúdo íntegro).
    """
    pw, ph = float(page.rect.width), float(page.rect.height)
    for z in itens:
        a = np.asarray(z["box"], float)
        x0 = float(a[:, 0].min())
        altura_box = float(np.ptp(a[:, 1]))
        fs = max(4.0, altura_box * 0.75)
        x = min(max(x0, 1.0), pw - 12.0)
        y = min(max(float(a[:, 1].max()) - 1.0, fs), ph - 2.0)
        passo = fs * 1.15
        largura_util = max(30.0, pw - x - 4.0)
        max_chars = max(8, int(largura_util / (fs * 0.55)))  # largura média conservadora
        for linha in _quebrar(_sanitizar_camada(z["texto"]), max_chars):
            page.insert_text(fitz.Point(x, y), linha, fontsize=fs, render_mode=3, overlay=True)
            y += passo
            if y > ph - 2.0:
                y = fs  # recomeça no topo; coordenadas permanecem dentro da página


def _reidratar_jpgs(r: dict, cfg: ConfigV3) -> None:
    """Recria buffers JPEG de um resultado vindo do cache re-renderizando a origem.

    O pré-processamento é determinístico e a assinatura cobre toda a configuração
    relevante, então as caixas armazenadas continuam batendo com as imagens.
    """
    faltando = {p["pagina"] for p in r["paginas"] if "jpg" not in p}
    if not faltando:
        return
    origem = Path(r["arquivo_origem"])
    if not origem.exists():
        LOG.warning("%s sumiu; páginas em cache ficarão em branco no PDF", origem.name)
        return
    try:
        recuperados = {}
        for n, img, _dpi in _fonte_paginas(origem, cfg):
            if n in faltando:
                usado = _pre(img, cfg)
                ok, buf = cv2.imencode(
                    ".jpg", usado, [int(cv2.IMWRITE_JPEG_QUALITY), int(cfg.jpeg_qualidade)]
                )
                if ok:
                    recuperados[n] = (buf.tobytes(), int(usado.shape[1]), int(usado.shape[0]))
        for p in r["paginas"]:
            if "jpg" not in p and p["pagina"] in recuperados:
                jpg, w, h = recuperados[p["pagina"]]
                p.update(jpg=jpg, largura_px=w, altura_px=h)
    except Exception:
        LOG.exception("Falha ao reidratar imagens de %s", origem.name)


def exportar_v3(r: dict, cfg: ConfigV3) -> None:
    base = Path(cfg.saida) / r["id"]

    _atomic_json(base.with_suffix(".json"), _leve(r))

    buf_csv = io.StringIO()
    escritor = csv.writer(buf_csv)
    escritor.writerow(["arquivo", "pagina", "texto", "score", "box"])
    for p in r["paginas"]:
        for z in p["itens"]:
            escritor.writerow(
                [r["arquivo_origem"], p["pagina"], z["texto"], z["score"], json.dumps(z["box"])]
            )
    _atomic_bytes(base.with_suffix(".csv"), buf_csv.getvalue().encode("utf-8"))

    if not cfg.gerar_pdf_pesquisavel or not r["paginas"]:
        return

    _reidratar_jpgs(r, cfg)  # BUG-1: cache quente continua gerando o PDF completo

    doc = fitz.open()
    try:
        for p in r["paginas"]:
            jpg = p.get("jpg")
            if jpg:
                page = doc.new_page(width=p["largura_px"], height=p["altura_px"])
                page.insert_image(page.rect, stream=jpg)
                _inserir_camada_texto(page, p["itens"])
            else:
                page_w, page_h = _TAMANHO_PLACEHOLDER
                doc.new_page(width=page_w, height=page_h)  # preserva a numeração
                LOG.warning(
                    "%s: página %d sem imagem (erro registrado); inserida página em branco",
                    r["arquivo_origem"],
                    p["pagina"],
                )
        dados_pdf = doc.tobytes(garbage=3, deflate=True)
    finally:
        doc.close()
    _atomic_bytes(base.with_name(base.name + "_pesquisavel.pdf"), dados_pdf)


# --------------------------------------------------------------------------- #
# Orquestração
# --------------------------------------------------------------------------- #

def executar_v3(
    cfg: ConfigV3 = CFG3,
    motor: MotorOCRV3 | None = None,
    progresso: ProgressoCallback | None = None,
) -> list[dict]:
    entrada, saida = Path(cfg.entrada), Path(cfg.saida)
    entrada.mkdir(parents=True, exist_ok=True)
    saida.mkdir(parents=True, exist_ok=True)

    arquivos = sorted(
        p for p in entrada.rglob("*")
        if p.is_file() and p.suffix.lower() in cfg.extensoes
    )
    if not arquivos:
        print(f"Nenhum arquivo em {entrada.resolve()}")
        return []

    def _avisar(atual: int, total_u: int, msg: str) -> None:
        """Callback protegido: falha do consumidor nunca derruba o pipeline."""
        if progresso is None:
            return
        try:
            progresso(atual, total_u, msg)
        except Exception:
            LOG.debug("Callback de progresso falhou (ignorado)", exc_info=True)

    # Unidades de progresso = páginas (reais quando o PDF é legível; 1 caso contrário).
    total_unidades = sum((_conta_paginas(p) or 1) for p in arquivos)
    _avisar(0, total_unidades, f"Preparando {len(arquivos)} documento(s)")

    if motor is None:
        try:
            motor = MotorOCRV3(cfg)
        except Exception:
            if cfg.backend == "rapidocr":
                raise
            LOG.exception("Backend '%s' falhou ao iniciar; caindo para 'rapidocr'", cfg.backend)
            cfg = replace(cfg, backend="rapidocr")
            motor = MotorOCRV3(cfg)

    resultados: list[dict] = []
    inicio_lote = time.perf_counter()
    feitas = 0
    for path in tqdm(arquivos, desc="documentos"):
        try:

            def _cb_pagina(atual: int, tot: int, msg: str, _base=feitas) -> None:
                _avisar(_base + atual, total_unidades, msg)

            r = processar_v3(path, cfg, motor, progresso=_cb_pagina)
            exportar_v3(r, cfg)
            for p in r["paginas"]:  # BUG-2: nada de imagem acumulada nos resultados
                p.pop("jpg", None)
            feitas += len(r["paginas"])
            resultados.append(r)
            origem = "cache" if r.get("_do_cache") else "ocr   "
            print(
                f"OK [{origem}] {path.name}: {len(r['paginas'])} páginas, "
                f"{len(r['erros'])} com erro, {r['tempo_s']:.2f}s"
            )
            _avisar(feitas, total_unidades, f"Exportado {path.name} ({len(r['paginas'])} pág.)")
        except Exception:
            LOG.exception("Documento ignorado: %s", path)

    _avisar(total_unidades, total_unidades, "Concluído")

    duracao = round(time.perf_counter() - inicio_lote, 2)
    tot_paginas = sum(len(x["paginas"]) for x in resultados)
    tot_erros = sum(len(x["erros"]) for x in resultados)
    sem_texto = sum(1 for x in resultados for p in x["paginas"] if not p["itens"])
    _atomic_json(
        saida / "resumo.json",
        {
            "versao": _VERSAO,
            "gerado_em": datetime.now().isoformat(timespec="seconds"),
            "duracao_total_s": duracao,
            "documentos": len(resultados),
            "paginas": tot_paginas,
            "erros": tot_erros,
            "paginas_sem_texto": sem_texto,
            "config": asdict(cfg),
        },
    )
    print(
        f"Concluído: {len(resultados)} documento(s), {tot_paginas} página(s), "
        f"{tot_erros} com erro, {sem_texto} sem texto detectado — {duracao:.1f}s"
    )
    return resultados

# Uso no notebook: coloque arquivos em ./documentos e rode:
#   resultados_v3 = executar_v3(CFG3)


## Execução

### Pipeline (lote)

1. Coloque PDFs e imagens em `./documentos`.
2. Rode a célula de implementação acima.
3. Rode `resultados_v3 = executar_v3(CFG3)` (ou com callback: `executar_v3(cfg, progresso=lambda atual, total, msg: print(f"{atual}/{total} {msg}"))`).
4. Cada documento gera JSON estruturado, CSV de caixas e PDF pesquisável em `./saida_ocr_v3`.

### Interface web local (Gradio)

Rode a célula final deste notebook. Ela constrói a interface e **não sobe servidor
sozinho**; para abrir, defina `OCR_V3_UI_LAUNCH=1` antes (ou rode o comando impresso):

```python
app.queue(default_concurrency_limit=1).launch(
    share=False, inbrowser=False, server_name="127.0.0.1",
)
```

A interface tem upload com prévia, seleção de páginas, abas Texto/Tabela/Fórmula,
progresso real, downloads (TXT/JSON/CSV/PDF pesquisável) e tema claro branco/rosa.
Tudo permanece local (`share=False`, loopback).

## Correções V2 → V3 (resumo honesto)

| Achado | Correção |
|---|---|
| 🔴 Cache quente sobrescrevia o PDF pesquisável com um PDF vazio | `_reidratar_jpgs()` re-renderiza as páginas antes de exportar |
| 🔴 Todo o lote ficava em RAM (~26 MB/página cru) | páginas viram buffer JPEG (~1–2 MB) e buffers morrem após exportar |
| 🟠 Schema PaddleOCR 3.x aninhado (`{"res": ...}`) causava erro em toda página | desembrulhamento defensivo + atributos |
| 🟠 `engine="onnxruntime"` podia não existir na sua versão | tentativa com fallback `TypeError` e degradação graciosa |
| 🟠 Deskew intermitente no OpenCV ≥ 4.5 (convenção do minAreaRect mudou) | `_medir_skew()` normalizado pelo lado maior |
| 🟠 `insert_textbox` descartava texto invisível que não cabia | re-fluxo de texto dentro da página (`_inserir_camada_texto`) |
| 🟡 Páginas com erro sumiam do PDF (numeração mudava) | placeholder em branco preserva numeração |
| 🟡 Temporários vazavam em exceção; flags de export invalidavam cache; ID dependia do caminho; `cfg.saida` ignorada | try/finally, assinatura só de inferência, ID por conteúdo, saída respeitada |
| 🟡 Falha no backend derrubava o lote inteiro | fallback automático para rapidocr (novo padrão) |

### Limites

Valide uma amostra representativa antes do lote. O PDF pesquisável contém a mesma imagem
pré-processada analisada pelo OCR; caixas e camada invisível compartilham o mesmo sistema
de coordenadas. Para manuscrito e tabelas complexas, mantenha revisão humana ou um módulo
separado de estruturação. Ordem de leitura por bandas: colunas muito complexas podem exigir
análise própria. Extração tabular é experimental; fórmulas não estão implementadas;
processamento sequencial por design (16 GB de RAM).


In [ ]:
# Interface web LOCAL (Gradio) — upload, prévia, abas Texto/Tabela/Fórmula.
# Nenhum documento sai da máquina: share=False e bind em 127.0.0.1.
import os

from ocr_v3.ui_gradio import criar_ui

app = criar_ui(CFG3)

if os.environ.get("OCR_V3_UI_LAUNCH") == "1":
    # Abre a interface de verdade (bloqueia a célula enquanto o servidor rodar).
    app.queue(default_concurrency_limit=1).launch(
        share=False,
        inbrowser=False,
        server_name="127.0.0.1",
    )
else:
    # Construção SEM servidor (padrão): seguro para executar/re-executar o notebook.
    print("Interface construída — servidor NÃO iniciado.")
    print("Para abrir a UI, rode numa célula:")
    print('    app.queue(default_concurrency_limit=1).launch(share=False, inbrowser=False, server_name="127.0.0.1")')
